In [1]:
import io
import json
import random
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from datasets import load_from_disk

PROJECT_ROOT = Path.cwd().parent.parent


In [4]:
with open(PROJECT_ROOT / 'polyvore_dataset/valid_no_dup_cleaned.json') as f:
    data = json.load(f)

ds = load_from_disk(str(PROJECT_ROOT / "polyvore_dataset/hf_images"))

id_to_idx = {item_id: i for i, item_id in enumerate(ds["item_ID"])}

def get_image(item_id):
    idx = id_to_idx.get(item_id)
    return ds[idx]["image"] if idx is not None else None

out = widgets.Output()

def show_item(item_id, item):
    colors = item.get('dominant_color', [0,0,0]), item.get('secondary_color', [0,0,0])

    img_pil = get_image(item_id)
    if img_pil is None:
        print(f'Unable to display item with id {item_id}')
        return

    try:
        buf = io.BytesIO()
        img_pil.save(buf, format='PNG')
        img_widget = widgets.Image(value=buf.getvalue(), format='png')
    except Exception as e:
        img_widget = widgets.Label(str(e))

    swatches = "".join(
        f'<div style="display:flex; align-items:center; margin:4px 0">'
        f'  <div style="width:40px; height:40px; border-radius:6px; background:rgb({c[0]},{c[1]},{c[2]}); margin-right:10px"></div>'
        f'  <code>rgb({c[0]},{c[1]},{c[2]})</code>'
        f'</div>'
        for c in colors
    )
    colors_widget = widgets.HTML(f"<b>ID: {item_id}, Name: {item.get('name', 'unknown')}</b><br><br>{swatches}")
    with out:
        clear_output(wait=True)
        display(widgets.HBox([img_widget, colors_widget]))


def get_random_item():
    outfit = random.choice(data)
    return outfit, random.choice(outfit['items'])


outfit, item = get_random_item()
item_id = f"{outfit['set_id']}_{item['index']}"
print(f"Loaded: item_id={item_id}")


Loaded: item_id=168989974_6


In [5]:
btn = widgets.Button(description="Next random item")

def on_next(b):
    global outfit, item, item_id
    outfit, item = get_random_item()
    item_id = f"{outfit['set_id']}_{item['index']}"
    show_item(item_id, item)

btn.on_click(on_next)
display(btn, out)
show_item(item_id, item)


Button(description='Next random item', style=ButtonStyle())

Output()